# PolyChain: Polymer Property Prediction

**Architecture:** GIN-S backbone + HAMF + PECGN + CST

**Run time:** ~30-60 min for full 5-fold CV on T4 GPU.

In [ ]:
#@title 1. Mount Drive + Clone + Install (run this first, always)
import os, sys

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_PATH = "/content/drive/MyDrive/PolyChain"
os.makedirs(f"{DRIVE_PATH}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_PATH}/predictions", exist_ok=True)
os.makedirs(f"{DRIVE_PATH}/reports", exist_ok=True)

# Clone repo (skip if already cloned)
if not os.path.exists("/content/Poly/polymer_competition"):
    !git clone https://github.com/NotShubham1112/Poly.git /content/Poly 2>/dev/null

os.chdir("/content/Poly/polymer_competition")

# Add project root to Python path
PROJECT_ROOT = "/content/Poly/polymer_competition"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Install dependencies
!pip install -q torch torchvision torch-geometric
!pip install -q rdkit xgboost lightgbm catboost
!pip install -q scikit-learn pandas numpy scipy pyyaml tqdm
!pip install -q matplotlib seaborn joblib

# Verify
import torch
import rdkit
import torch_geometric
print(f"PyTorch: {torch.__version__}")
print(f"PyG: {torch_geometric.__version__}")
print(f"RDKit: {rdkit.__version__}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} ({props.total_memory / 1e9:.1f} GB)")
else:
    print("GPU: not available (using CPU)")
print(f"Project root: {PROJECT_ROOT}")
print("Setup complete!")

In [ ]:
#@title 2. Verify Project Imports
import os, sys, json, pickle, time
import numpy as np
import pandas as pd
import yaml

# Ensure project root is in path
PROJECT_ROOT = "/content/Poly/polymer_competition"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from features.graphs import smiles_to_graph, kmer_graph, periodic_graph, BOND_FEAT_DIM
from features.graph_utils import build_multiscale, collate_multiscale
from models.polychain import PolyChain
from models.polychain.cst import compute_cst_batch, CST_DIM
from models.gnn import get_gnn
from models.mlp import FingerprintMLP
from training.train_utils import rmse, mae, r2_score, spearman, save_checkpoint, load_checkpoint
from reports.visualizations import ReportGenerator

print(f"BOND_FEAT_DIM: {BOND_FEAT_DIM}")
print(f"CST_DIM: {CST_DIM}")
print("All imports successful!")

In [ ]:
#@title 3. Load or Download Dataset
import os, pandas as pd

DATA_DIR = "data"
train_path = os.path.join(DATA_DIR, "train.csv")

if not os.path.exists(train_path):
    print("train.csv not found. Generating polymer Tg dataset...")
    !python data/download_polymer_data.py --dataset polymer_tg --data_dir {DATA_DIR}

train_df = pd.read_csv(train_path)
print(f"Dataset: {len(train_df)} samples")
print(f"Columns: {list(train_df.columns)}")
if "property" in train_df.columns:
    y = train_df["property"]
    print(f"Target: mean={y.mean():.3f}, std={y.std():.3f}, min={y.min():.3f}, max={y.max():.3f}")
train_df.head()

In [ ]:
#@title 4. Visualize Target Distribution
import os, sys, pandas as pd
sys.path.insert(0, "/content/Poly/polymer_competition")
from reports.visualizations import ReportGenerator

gen = ReportGenerator("reports/plots")
train_df = pd.read_csv("data/train.csv")
if "property" in train_df.columns:
    gen.plot_target_distribution(train_df["property"].values, target_name="property")

In [ ]:
#@title 5. Generate CV Splits & Features
!python -m data.generate_splits --config config.yaml
!python -m features.build_features --config config.yaml

In [ ]:
#@title 6. Smoke Test: 100 Polymers, 3 Epochs
!python -m training.train --model_type polychain --fold 0 --person colab --max_samples 100 --epochs 3

In [ ]:
#@title 7. Verify Checkpoint + Inference
import os, sys
sys.path.insert(0, "/content/Poly/polymer_competition")
from inference.predictor import PolymerPredictor

ckpt_path = "outputs/checkpoints/colab_polychain_fold0_best.pt"
if os.path.exists(ckpt_path):
    predictor = PolymerPredictor(ckpt_path)
    test_smiles = ["*CCO*", "*CC(*)c1ccccc1*", "*CCl*"]
    preds = predictor.predict(test_smiles)
    for smi, pred in zip(test_smiles, preds):
        print(f"  {smi} -> {pred:.4f}")
else:
    print(f"Checkpoint not found: {ckpt_path}")
    print("Run cell 6 first.")

In [ ]:
#@title 8. Copy Checkpoint to Drive
import os
DRIVE_PATH = "/content/drive/MyDrive/PolyChain"
!cp -v outputs/checkpoints/colab_polychain_fold0_*.pt "$DRIVE_PATH/checkpoints/" 2>/dev/null || echo "No checkpoints to copy"
print("Drive checkpoints:", os.listdir(f"{DRIVE_PATH}/checkpoints/") if os.path.exists(f"{DRIVE_PATH}/checkpoints/") else [])

In [ ]:
#@title 9. Full 5-Fold CV (all models)
#@markdown Select models to train:
MODELS = "ridge,rf,xgb,gcn,gat,polychain" #@param {type:"string"}

!python -m training.run_all_folds --models "$MODELS" --person colab --config config.yaml

In [ ]:
#@title 10. View Results Summary
import os, pandas as pd
summary_path = "results/summary.csv"
if os.path.exists(summary_path):
    summary = pd.read_csv(summary_path)
    print("=" * 60)
    print("  MODEL COMPARISON (5-Fold CV)")
    print("=" * 60)
    print(summary.to_string(index=False))
else:
    print("No summary found. Run cell 9 first.")

In [ ]:
#@title 11. Model Comparison Plot
import os, sys, pandas as pd
sys.path.insert(0, "/content/Poly/polymer_competition")
from reports.visualizations import ReportGenerator

gen = ReportGenerator("reports/plots")
summary_path = "results/summary.csv"
if os.path.exists(summary_path):
    summary = pd.read_csv(summary_path)
    model_rmse = dict(zip(summary["model"], summary["rmse_mean"]))
    gen.plot_model_comparison(model_rmse, metric_name="rmse")
else:
    print("No summary found. Run cell 9 first.")

In [ ]:
#@title 12. PolyChain Ablation Study
!python -m training.run_ablation --fold 0 --epochs 50 --config config.yaml

In [ ]:
#@title 13. View Ablation Results
import os, sys, pandas as pd
sys.path.insert(0, "/content/Poly/polymer_competition")
from reports.visualizations import ReportGenerator

gen = ReportGenerator("reports/plots")
ablation_path = "results/ablation_results.csv"
if os.path.exists(ablation_path):
    ablation = pd.read_csv(ablation_path)
    print("=" * 60)
    print("  POLYCHAIN ABLATION STUDY")
    print("=" * 60)
    print(ablation.to_string(index=False))
    variant_rmse = dict(zip(ablation["variant"], ablation["rmse"]))
    gen.plot_ablation(variant_rmse)
else:
    print("No ablation results. Run cell 12 first.")

In [ ]:
#@title 14. Generate Reports
!python reports/generate_reports.py --config config.yaml --skip-shap

In [ ]:
#@title 15. Display Generated Plots
import os
plot_dir = "reports/plots"
if os.path.exists(plot_dir):
    plots = sorted([f for f in os.listdir(plot_dir) if f.endswith(".png")])
    print(f"Generated {len(plots)} plots:")
    for p in plots:
        print(f"  - {p}")
else:
    print("No plots generated yet.")

In [ ]:
#@title 16. Save Everything to Drive
import os
DRIVE_PATH = "/content/drive/MyDrive/PolyChain"
!cp -rv outputs/checkpoints/* "$DRIVE_PATH/checkpoints/" 2>/dev/null || true
!cp -rv predictions/* "$DRIVE_PATH/predictions/" 2>/dev/null || true
!cp -rv reports/* "$DRIVE_PATH/reports/" 2>/dev/null || true
!cp -rv results/* "$DRIVE_PATH/reports/" 2>/dev/null || true
print(f"All results synced to: {DRIVE_PATH}")

In [ ]:
#@title 17. Resume After Disconnection
import os
DRIVE_PATH = "/content/drive/MyDrive/PolyChain"
!cp -v "$DRIVE_PATH/checkpoints/"*.pt outputs/checkpoints/ 2>/dev/null || echo "No checkpoints in Drive"
!python -m training.train --model_type polychain --fold 0 --person colab --config config.yaml --resume

In [ ]:
#@title 18. Inference Demo
import os, sys, glob
sys.path.insert(0, "/content/Poly/polymer_competition")
from inference.predictor import PolymerPredictor

best_ckpts = glob.glob("outputs/checkpoints/*polychain*best.pt")
if best_ckpts:
    predictor = PolymerPredictor(best_ckpts[0])
    test_polymers = {
        "Polyethylene (PE)": "*CC*",
        "Polystyrene (PS)": "*CC(*)c1ccccc1*",
        "PVC": "*CCl*",
        "PMMA": "*CC(=O)OC*",
        "Nylon 6,6": "*CC(=O)NCCCCCC(=O)N*",
    }
    print(f"{'Polymer':<20} {'SMILES':<30} {'Predicted':>10}")
    print("-" * 62)
    for name, smi in test_polymers.items():
        pred = predictor.predict([smi])
        print(f"{name:<20} {smi:<30} {pred[0]:>10.2f}")
else:
    print("No checkpoint found. Run cell 6 first.")